# Задание 6: Распознавание лица и подсчет пальцев пользователя

In [ ]:
import cv2
import mediapipe as mp
import face_recognition
import os

def calculate_active_fingers(landmarks):
    finger_tips = [4, 8, 12, 16, 20]
    active = 0
    
    if landmarks.landmark[finger_tips[0]].x < landmarks.landmark[finger_tips[0] - 1].x:
        active += 1
        
    for idx in range(1, 5):
        if landmarks.landmark[finger_tips[idx]].y < landmarks.landmark[finger_tips[idx] - 2].y:
            active += 1
            
    return active

def run_system():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Camera could not be opened.")
        return

    known_enc = None
    print("Press 'c' when looking at the camera to calibrate your face as the 'owner'.")
    print("Press 'q' at any time to exit.")
    
    # 1. Capture Face from Camera instead of a file
    while known_enc is None:
        success, frame = cap.read()
        if not success:
            break
            
        display_frame = frame.copy()
        cv2.putText(display_frame, "Press 'c' to capture owner face", (20, 40), cv2.FONT_HERSHEY_PLAIN, 1.5, (0, 255, 255), 2)
        cv2.imshow("Stream", display_frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c'):
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            encodings = face_recognition.face_encodings(rgb)
            if encodings:
                known_enc = encodings[0]
                print("Owner face registered! Starting system...")
            else:
                print("No face detected! Please ensure your face is visible and press 'c' again.")
        elif key == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            return
            
    # 2. Main Processing Loop
    mp_hands = mp.solutions.hands
    detector = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7)
    drawer = mp.solutions.drawing_utils
    
    while True:
        success, frame = cap.read()
        if not success:
            break
            
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        face_boxes = face_recognition.face_locations(rgb)
        encodings = face_recognition.face_encodings(rgb, face_boxes)
        
        owner_present = False
        
        for enc, box in zip(encodings, face_boxes):
            top, right, bottom, left = box
            is_match = face_recognition.compare_faces([known_enc], enc, tolerance=0.55)[0]
            
            color = (255, 0, 0) if is_match else (0, 0, 255)
            label = "ME" if is_match else "GUEST"
            owner_present = owner_present or is_match
            
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.putText(frame, label, (left, top - 10), cv2.FONT_HERSHEY_PLAIN, 1, color, 2)
            
        if owner_present:
            hands_result = detector.process(rgb)
            total_fingers = 0
            if hands_result.multi_hand_landmarks:
                for hand_landmarks in hands_result.multi_hand_landmarks:
                    drawer.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    total_fingers += calculate_active_fingers(hand_landmarks)
            cv2.putText(frame, f"Fingers Detected: {total_fingers}", (30, 40), cv2.FONT_HERSHEY_PLAIN, 2, (0, 255, 0), 3)
        else:
            cv2.putText(frame, "Waiting for owner...", (30, 40), cv2.FONT_HERSHEY_PLAIN, 1.5, (0, 0, 255), 2)
            
        cv2.imshow("Stream", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_system()

c:\Users\livee\miniconda3\envs\torch_course\lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Press 'c' when looking at the camera to calibrate your face as the 'owner'.
Press 'q' at any time to exit.


RuntimeError: Unsupported image type, must be 8bit gray or RGB image.

: 